**Ejercicio 1**

1. Fichero de Entrada (Input):
Un único fichero de texto (quijote.txt) que contiene la obra completa.

2. Fase MAP (Mapper):
El mapper debe leer el fichero línea por línea y realizar las siguientes tareas de normalización antes de emitir los pares clave-valor:

3. Fase REDUCE (Reducer):
El reducer recibirá una palabra (clave) y una lista de ‘1’s (valores) asociados a esa palabra.

4. Fichero de Salida (Output):
El resultado final debe ser un fichero en formato CSV donde cada línea contenga una palabra y su número total de ocurrencias.

In [ ]:
!head -n 8 quijote.txt

In [ ]:
!hdfs dfs -put /media/notebooks/quijote.txt /


In [ ]:
!hdfs dfs -ls /


In [ ]:
!hdfs dfs -head /quijote.txt


In [ ]:
%%writefile mapperQuijote.py
#!/usr/bin/env python3
import sys

lista = ["¿", "?", "¡", "!", ".", ",", ";", ":", "(", ")", '"', "'", "-"]

for line in sys.stdin:

    line = line.strip()
    for char in lista:
        line = line.replace(char, "")
    words = line.split()

    for word in words:
        print(f"{word.lower()}\t1")

In [ ]:
%%writefile reducerQuijote.py
#!/usr/bin/env python3
import sys

palabra = None
contador = 0
for line in sys.stdin:
    line = line.strip()
    clave, valor = line.split()
    valor = int(valor)

    if palabra is None:
        palabra = clave
        contador = valor
    if clave == palabra:
        contador += valor
    else:
        print(f"{palabra},{contador}")
        palabra = clave
        contador = valor

# Última línea
if palabra is not None:
    print(f"{palabra},{contador}")

In [ ]:
#Simulamos
!head -n 2 quijote.txt | python3 mapperQuijote.py | sort | python3 reducerQuijote.py

Lo ejecutamos en Streaming

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapperQuijote.py \
-file reducerQuijote.py \
-mapper mapperQuijote.py \
-reducer reducerQuijote.py \
-input /quijote.txt \
-output /Quijote

In [ ]:
!hdfs dfs -head  /Quijote/part-00000

**Ejercicio 2**

Crear una lista (o carga desde un fichero) de palabras no significativas comunes en español (ej. “de”, “la”, “el”, “y”, “en”, “que”, “a”, “los”, “del”, “se”).

Modificar el mapper para que no emita ningún par clave-valor si la palabra se encuentra en tu lista de palabras no significativas.

El resultado final será un conteo de palabras significativas, excluyendo las más comunes y menos informativas.


In [ ]:
%%writefile mapperQuijote2.py
#!/usr/bin/env python3
import sys
lista2 = ["de", "la", "el", "y", "en", "que", "a", "los", "del", "se"]
lista = ["¿", "?", "¡", "!", ".", ",", ";", ":", "(", ")", '"', "'", "-",]
lista_aux = []
for line in sys.stdin:

    line = line.strip()
    for char in lista:
        line = line.replace(char, "")
    words = line.split()
    lista_aux = []
    for palabra in words:
        if palabra.lower() not in lista2:
            lista_aux.append(palabra.lower())

    for palabrasLista in lista_aux:
        print(f"{palabrasLista.lower()}\t1")

In [ ]:
%%writefile reducerQuijote2.py
#!/usr/bin/env python3
import sys

palabra = None
contador = 0
for line in sys.stdin:
    line = line.strip()
    clave, valor = line.split("\t")
    valor = int(valor)

    if palabra is None:
        palabra = clave
        contador = valor
    if clave == palabra:
        contador += valor
    else:
        print(f"{palabra},{contador}")
        palabra = clave
        contador = valor

# Última línea
if palabra is not None:
    print(f"{palabra},{contador}")

In [ ]:
!head -n 2 quijote.txt | python3 mapperQuijote2.py | sort | python3 reducerQuijote2.py

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapperQuijote2.py \
-file reducerQuijote2.py \
-mapper mapperQuijote2.py \
-reducer reducerQuijote2.py \
-input /quijote.txt \
-output /Quijote2

In [ ]:
!hdfs dfs -ls /Quijote2

In [ ]:
!hdfs dfs -head /Quijote2/part-00000

**Ejercicio 3**

Implementar un segundo trabajo de MapReduce que tome la salida del primero.

Este segundo trabajo debe reordenar los datos para que la salida final esté ordenada por frecuencia de forma ascendente, mostrando las palabras más usadas primero.

Tienes que aprovecharte de que Hadoop se encarga de ordenar por la clave, así que en el mapper del segundo trabajo deberías invertir el par (clave, valor) para que sea (valor, clave)


In [ ]:
%%writefile mapperQuijote3.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    for word in line.strip().split():
        clean_word = ""
        for char in word:
            if char.isalpha():
                clean_word += char

        if clean_word:
            print(f"{clean_word}\t1")

In [ ]:
!head -n 1 quijote.txt | python3 mapperQuijote3.py 

In [ ]:
%%writefile mapperQuijote3_2.py
#!/usr/bin/env python3
import sys

word_dict = {}
for line in sys.stdin:
    line = line.strip()
    clave, valor = line.split("\t")
    valor = int(valor)
    if clave not in word_dict:
        word_dict[clave] = valor
    else:
        word_dict[clave] += valor

for key, value in sorted(word_dict.items(), key=lambda x: x[1], reverse=True):
    print(f"{value}\t{key}")

In [ ]:
!head -n 2 quijote.txt | python3 mapperQuijote3.py |  python3 mapperQuijote3_2.py

Hacemos el Primer mapper

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-D mapreduce.job.reduces=0 \
-file mapperQuijote3.py \
-mapper mapperQuijote3.py \
-input /quijote.txt \
-output /Quijote3

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-D mapreduce.job.reduces=0 \
-file mapperQuijote3_2.py \
-mapper mapperQuijote3_2.py \
-input /Quijote3/part-00000\
-output /Quijote3_2

In [ ]:
!hdfs dfs -head /Quijote3_2/part-00000